# Part 2: Denoising Autoencoder

### Extend the Autoencoder you implemented in Part 1 to a Denoising Autoencoder

Recall from the lecture, a denoising autoencoder's architecture is very similar to a standard autoencoder. The difference is the input to the autoencoder has noise added to it. However, when computing the loss function, make sure the original (non-noisy) version is used for backpropagation.

Again, let's start by loading the Fashion-MNIST dataset and transforming it to a flattened tensor.

In [1]:
%matplotlib inline

import torch
import torchvision
import torchvision.transforms as transforms

import numpy as np

batch_size = 256
image_dim = 784  # [flattened]

# dataset construction
transform = transforms.Compose([
    transforms.ToTensor(), # convert to tensor
    transforms.Lambda(lambda x: x.view(image_dim)) # flatten into vector
    ])

train_set = torchvision.datasets.FashionMNIST(
    root='./data/FashionMNIST'
    ,train=True
    ,download=True
    ,transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=batch_size
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 20.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 340kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 6.33MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.8MB/s]


## Build a Denoising Autoencoder

Now, define the Encoder and Decoder classes for your denoising autoencoder, called DN_Encoder, DN_Decoder, respectively. You can define these architectures how you like; some suggested architectures are given as comments in the classes below.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from tqdm import tqdm
from itertools import chain


class DN_Encoder(nn.Module):
    '''
    Denoising encoder with a single input, hidden and output layer
    '''
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DN_Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        z = self.fc2(x)
        return z


class DN_Decoder(nn.Module):
    '''
    Denoising decoder: single dense hidden layer followed by
    output layer with a sigmoid to squish values
    '''
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DN_Decoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x_recon = torch.sigmoid(self.fc2(x))
        return x_recon

## Learning your Denoising Autoencoder

Start from the training procedure used in Part 1 for the autoencoder and extend this to get your denoising autoencoder working. Again, include images of both the data with added noise as well as the reconstructed images in the submitted notebook. Regarding the noise to add to your images, add Gaussian noise with a mean of 0 and a standard deviation of 1.

In [3]:
# YOUR CODE HERE
# hyperparameters
input_dim = image_dim
hidden_dim = 256
latent_dim = 64
nEpoch = 10
noise_std = 1.0

# construct encoder, decoder and optimiser
dn_enc = DN_Encoder(input_dim, hidden_dim, latent_dim)
dn_dec = DN_Decoder(latent_dim, hidden_dim, input_dim)

optimizer = optim.Adam(
    chain(dn_enc.parameters(), dn_dec.parameters()),
    lr=1e-3
)

# training loop
for epoch in range(nEpoch):
    losses = []
    trainloader = tqdm(train_loader)

    for i, data in enumerate(trainloader, 0):
        inputs, _ = data

        # add Gaussian noise: mean = 0, std = 1
        noisy_inputs = inputs + noise_std * torch.randn_like(inputs)

        # because image pixels are in [0, 1], clip noisy images back to [0, 1]
        noisy_inputs = torch.clamp(noisy_inputs, 0., 1.)

        optimizer.zero_grad()

        # input is noisy image
        z = dn_enc(noisy_inputs)
        outputs = dn_dec(z)

        # target is original clean image
        loss = F.binary_cross_entropy(outputs, inputs, reduction='sum') / inputs.shape[0]

        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        trainloader.set_postfix(loss=np.mean(losses), epoch=epoch)


# display noisy and reconstructed images
dn_enc.eval()
dn_dec.eval()

dataiter = iter(train_loader)
images, labels = next(dataiter)

with torch.no_grad():
    noisy_images = images + noise_std * torch.randn_like(images)
    noisy_images = torch.clamp(noisy_images, 0., 1.)

    z = dn_enc(noisy_images)
    reconstructed = dn_dec(z)

n = 8
plt.figure(figsize=(12, 6))

for i in range(n):
    # original clean images
    plt.subplot(3, n, i + 1)
    plt.imshow(images[i].view(28, 28), cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Original")

    # noisy images
    plt.subplot(3, n, i + 1 + n)
    plt.imshow(noisy_images[i].view(28, 28), cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Noisy")

    # reconstructed images
    plt.subplot(3, n, i + 1 + 2 * n)
    plt.imshow(reconstructed[i].view(28, 28), cmap='gray')
    plt.axis('off')
    if i == 0:
        plt.title("Reconstructed")

plt.show()

100%|██████████| 235/235 [00:13<00:00, 17.00it/s, epoch=9, loss=252]


NameError: name 'plt' is not defined